# Machine-learning classification and error analysis
Data: IBM Skills Network `dataset_part_2.csv` (90 rows). Remote API/Wikipedia collection is not claimed successful.


In [ ]:
import pandas as pd,numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,balanced_accuracy_score,recall_score,confusion_matrix,classification_report
x=pd.read_csv('dataset_part_2.csv');features=['PayloadMass','Flights','GridFins','Reused','Legs','Block','ReusedCount','Orbit','LaunchSite']
X=x[features].copy();y=x.Class.astype(int)
for c in X.select_dtypes('bool').columns:X[c]=X[c].astype(int)
num=X.select_dtypes(include='number').columns.tolist();cat=[z for z in features if z not in num]
pre=ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models={'Logistic Regression':LogisticRegression(max_iter=2000),'SVM RBF':SVC(),'Decision Tree':DecisionTreeClassifier(random_state=42),'KNN k=5':KNeighborsClassifier(5)}


In [ ]:
for name,model in models.items():
 pipe=Pipeline([('pre',pre),('model',model)])
 scores=cross_val_score(pipe,Xtr,ytr,cv=StratifiedKFold(5,shuffle=True,random_state=42))
 pipe.fit(Xtr,ytr);yp=pipe.predict(Xte)
 print(name,'CV',round(scores.mean(),3),'test',round(accuracy_score(yte,yp),3),'balanced',round(balanced_accuracy_score(yte,yp),3),'failure recall',round(recall_score(yte,yp,pos_label=0),3))
 print(confusion_matrix(yte,yp,labels=[0,1]))


Important interpretation: n=23 held-out cases includes 8 failures. The verified logistic-regression confusion matrix was `[[3,5],[0,15]]`: 78.3% accuracy but just 37.5% failure recall. A one-run KNN comparison reached 82.6% accuracy and 50.0% failure recall, but was weaker on mean training CV than SVM. Avoid claiming generalisation from one split.
